In [1]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gensim.models import Word2Vec
from sklearn.decomposition import PCA
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report


print("Libraries imported successfully")

Libraries imported successfully


In [2]:
df = pd.read_csv("/Users/theerdhasjoy/BIA/ML/Dataset/Twitter_Data.csv")
print("Shape : ", df.shape)
df.head()

Shape :  (27481, 4)


,textID,text,selected_text,sentiment
0,cb774db0d1,"I`d have responded, if I were going","I`d have responded, if I were going",neutral
1,549e992a42,Sooo SAD I will miss you here in San Diego!!!,Sooo SAD,negative
2,088c60f138,my boss is bullying me...,bullying me,negative
3,9642c003ef,what interview! leave me alone,leave me alone,negative
4,358bd9e861,"Sons of ****, why couldn`t they put them on t...","Sons of ****,",negative


In [3]:
df.dropna(inplace = True)

In [6]:
stemmer = PorterStemmer()

stop_words = set(stopwords.words("english"))

def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"[^a-zA-Z]", " ", text)

    words = text.split()

    words = [stemmer.stem(word) for word in words if word not in stop_words]

    return " ".join(words)


# Apply cleaning
df["clean_text"] = df["text"].apply(clean_text)

# Tokenization
df["tokens"] = df["clean_text"].apply(lambda x: x.split())

display(df[["text", "clean_text", "tokens"]].head())

,text,clean_text,tokens
0,"I`d have responded, if I were going",respond go,"[respond, go]"
1,Sooo SAD I will miss you here in San Diego!!!,sooo sad miss san diego,"[sooo, sad, miss, san, diego]"
2,my boss is bullying me...,boss bulli,"[boss, bulli]"
3,what interview! leave me alone,interview leav alon,"[interview, leav, alon]"
4,"Sons of ****, why couldn`t they put them on t...",son put releas alreadi bought,"[son, put, releas, alreadi, bought]"


In [7]:
model = Word2Vec(
    sentences= df["tokens"].tolist(),
    vector_size = 100,
    window= 5,
    min_count= 5,
    workers= 4,
    sg= 1,
    seed=42,
    epochs= 10
)

print("Word2Vec model is trained")
print("Vocabulary size : ", len(model.wv))


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec model is trained
Vocabulary size :  3821


In [8]:
print("First 30 words in vocabulary")
print(model.wv.index_to_key[:30])

First 30 words in vocabulary
['day', 'go', 'get', 'good', 'work', 'love', 'like', 'http', 'today', 'time', 'got', 'one', 'lol', 'happi', 'thank', 'want', 'u', 'know', 'miss', 'back', 'realli', 'see', 'com', 'im', 'think', 'feel', 'night', 'hope', 'mother', 'make']


In [9]:
for word in ["happy", "love", "good", "sad"]:
    if word in model.wv:
        print(f"\nWords similar to '{word}' : ")
        print(model.wv.most_similar(word, topn= 10))

    else:
        print(f"'{word}' is not in the vocabulary.")



'happy' is not in the vocabulary.

Words similar to 'love' : 
[('cullen', 0.730695903301239), ('brad', 0.7161985635757446), ('grate', 0.7139164805412292), ('ili', 0.7138152122497559), ('amaz', 0.7119051218032837), ('england', 0.7090522050857544), ('actress', 0.7069627046585083), ('sarah', 0.7066818475723267), ('coolest', 0.7060045599937439), ('sexi', 0.7036944627761841)]

Words similar to 'good' : 
[('great', 0.6933090090751648), ('tweep', 0.683656632900238), ('incred', 0.6836313605308533), ('fabul', 0.6756539940834045), ('ish', 0.673995852470398), ('hon', 0.6736157536506653), ('brunch', 0.6715958118438721), ('perhap', 0.6703631281852722), ('despit', 0.6690285801887512), ('workout', 0.6642637252807617)]

Words similar to 'sad' : 
[('tear', 0.688518762588501), ('greg', 0.671621561050415), ('lone', 0.6583543419837952), ('realiz', 0.6428197622299194), ('depress', 0.6414288878440857), ('teacher', 0.6370485424995422), ('graduat', 0.6362881660461426), ('pritchard', 0.6234839558601379), ('fas